In [1]:
import radiate as rd
import polars as pl
from IPython.display import display, HTML

rd.random.seed(67123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.linear()],
        edge=rd.Op.weight(),
        output=rd.Op.linear(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .alters(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-05-22T19:46:09.067587Z  INFO Epoch 1    | Score:   2.0038 | Time: 2.19ms
2026-05-22T19:46:09.067701Z  INFO Epoch 2    | Score:   2.0038 | Time: 2.25ms
2026-05-22T19:46:09.067778Z  INFO Epoch 3    | Score:   2.0038 | Time: 2.30ms
2026-05-22T19:46:09.067844Z  INFO Epoch 4    | Score:   1.6821 | Time: 2.34ms
2026-05-22T19:46:09.067921Z  INFO Epoch 5    | Score:   1.6821 | Time: 2.39ms
2026-05-22T19:46:09.068007Z  INFO Epoch 6    | Score:   1.6821 | Time: 2.44ms
2026-05-22T19:46:09.068097Z  INFO Epoch 7    | Score:   1.6821 | Time: 2.51ms
2026-05-22T19:46:09.068180Z  INFO Epoch 8    | Score:   1.6821 | Time: 2.56ms
2026-05-22T19:46:09.068263Z  INFO Epoch 9    | Score:   1.6821 | Time: 2.61ms
2026-05-22T19:46:09.068346Z  INFO Epoch 10   | Score:   1.6821 | Time: 2.67ms
2026-05-22T19:46:09.068442Z  INFO Epoch 11   | Score:   1.6821 | Time: 2.74ms
2026-05-22T19:46:09.068538Z  INFO Epoch 12   | Score:   1.6821 | Time: 2.81ms
2026-05-22T19:46:09.068616Z  INFO Epoch 13   | Score:   1.6821 |

In [9]:
# collector.plot()
print(result.metrics().dashboard())

[36 metrics, 50682 updates]
----- Metrics ----- (36 :: 50682) 
  carryover: 0.431  diversity: 0.571  unique_members: 66  unique_scores: 63  improvements: 114  iter_time(mean): 234.656µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt       | Entr      
-------------------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 1.000      | 0.000      | 3.000      | 100    | 0.100        | 1.239      | 3.298      | 15.559     | -         
scores                     | dist   | 0.586      | 0.001      | 11.376     | 100    | 0.059        | 2.047      | 6.692      | 39.938     | -         
size.genome                | dist   | 26.110     | 26.000     | 27.000     | 100    | 2.611        | 0.314      | 0.000      | 0.000      | -         


== Statistics ==
Name                    

In [5]:
eval_results = result.value().eval(inputs)
accuracy = rd.accuracy(result.value(), inputs, answers, loss=rd.MSE)
accuracy

PyAccuracy("Regression Accuracy" {
	N: 20 
	Accuracy: 99.66%
	R² Score: 0.99976
	RMSE: 0.03118
	Loss (MSE): 0.00097
})

In [6]:
df = collector.to_polars(lazy=False)
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",8.0,108.0,54.0,65.053825,4232.0,NaN,8.0,100.0,2,null,null,null,null,null,null,0,2,"[""statistic""]"
"""step.evaluate.time""",0.000253,0.001356,0.000678,0.000601,3.6121e-7,NaN,0.000253,0.001103,2,1356µs,678µs,601µs,253µs,1103µs,0µs,0,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,20.0,20.0,0.0,0.0,NaN,20.0,20.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000008,0.000008,0.000008,0.0,0.0,NaN,0.000008,0.000008,1,8µs,8µs,0µs,8µs,8µs,0µs,0,1,"[""selector"", ""time""]"
"""selector.tournament""",80.0,80.0,80.0,0.0,0.0,NaN,80.0,80.0,1,null,null,null,null,null,null,0,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""time""",0.000408,0.06148,0.000235,0.00022,4.8315e-8,0.0,0.000041,0.002187,262,61479µs,234µs,219µs,40µs,2187µs,0µs,261,1,"[""time""]"
"""index""",262.0,262.0,262.0,0.0,0.0,NaN,262.0,262.0,1,null,null,null,null,null,null,0,1,"[""statistic""]"
"""score.improvement""",1.0,114.0,1.0,0.0,0.0,0.0,1.0,1.0,114,null,null,null,null,null,null,261,1,"[""statistic"", ""score""]"


In [7]:
filtered = (
    df.filter(pl.col("name") == "score.improvement")
    .select("generation")
    .unique()
    .sort("generation")
)
filtered

generation
i64
0
3
96
102
103
…
255
257
258


In [8]:
display(HTML(filtered._repr_html_()))

generation
i64
0
3
96
102
103
…
255
257
258
